# Bull Market Support Band — early-entry (buy *below* the band) + fakeout filter
Classic BMSB only buys the breakout **above** the 20/21 band. This version adds an **anticipate** mode that starts buying **while price is still below the band** when momentum turns up (RSI rising from a low + an up-close), so we catch the move earlier — and a **fakeout filter** (a % stop + a 'must reclaim the band within N bars' timeout) to cut the fakeouts.

- Daily **and** weekly, 10 years, **$1,000/trade**.
- Sweeps band lengths, entry mode, RSI threshold, stop %, confirm bars; keeps only configs with **8–15 trades**; ranks by a blended **Score = Total P&L × Win Rate**.
- Weekly section also charts the **canonical 20 SMA / 21 EMA** band right next to the best one.

```bash
pip install yfinance pandas numpy matplotlib jupyter
```

In [ ]:
# ================= CONFIG — edit me =================
TICKER     = "AAPL"
YEARS      = 10
START_DATE = None        # e.g. "2018-01-01"  (overrides YEARS if set)
END_DATE   = None        # e.g. "2024-12-31"
CAPITAL    = 1000.0      # $ per trade
MIN_TRADES = 8           # only keep configs that trade at least this many times
MAX_TRADES = 15          # ...and at most this many (your 8-15 requirement)
RANK_BY    = "Score"     # "Score" (PnL x win-rate) | "Total P&L $" | "Win Rate %" | "Profit Factor"

# ---- BMSB early-entry engine knobs (defaults; the sweep overrides these) ----
ENTRY_MODE   = "anticipate"   # anticipate | reclaim | breakout | hybrid_e
RSI_BUY      = 45             # only "anticipate" below-band buys when RSI < this and rising
STOP_PCT     = 0.12          # fakeout stop: exit if price falls this % below entry
CONFIRM_BARS = 6             # exit if band not reclaimed within this many bars

# parameter grids to sweep
W_GRID = ([15,20,25], [21,30,40], ["anticipate","breakout","reclaim"], [40,45,50], [0.08,0.12], [4,6])
D_GRID = ([100,150,200,250], [100,150,200,250], ["anticipate","breakout"], [40,45,50], [0.08,0.12], [20,40])


In [ ]:
%matplotlib inline
import yfinance as yf
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30, "display.width", 200)


In [ ]:
def get_data(ticker, interval, years=YEARS, start=START_DATE, end=END_DATE):
    if start:
        df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=True, progress=False)
    else:
        df = yf.download(ticker, period=f"{years}y", interval=interval, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open","High","Low","Close","Volume"]].dropna().reset_index()
    df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    return df

daily  = get_data(TICKER, "1d")
weekly = get_data(TICKER, "1wk")
print(f"{TICKER}: {len(daily)} daily, {len(weekly)} weekly candles "
      f"({daily.Date.min().date()} -> {daily.Date.max().date()})")


In [ ]:
CAPITAL = float(CAPITAL)

def compute_stats(t, capital=CAPITAL):
    z = {"Total Trades":0,"Win Rate %":0,"Total P&L $":0,"Avg Win $":0,"Avg Loss $":0,
         "Profit Factor":0,"Expectancy $":0,"Avg Bars Held":0,"Max Drawdown $":0,"Return/Trade %":0}
    if t.empty: return z
    w = t[t.pnl>0]; l = t[t.pnl<=0]
    gp = w.pnl.sum(); gl = abs(l.pnl.sum())
    eq = t.pnl.cumsum(); dd = (eq-eq.cummax()).min()
    return {"Total Trades":len(t),
            "Win Rate %":round(len(w)/len(t)*100,1),
            "Total P&L $":round(t.pnl.sum(),2),
            "Avg Win $":round(w.pnl.mean(),2) if len(w) else 0.0,
            "Avg Loss $":round(l.pnl.mean(),2) if len(l) else 0.0,
            "Profit Factor":round(gp/gl,2) if gl>0 else float("inf"),
            "Expectancy $":round(t.pnl.mean(),2),
            "Avg Bars Held":round(t.bars.mean(),1),
            "Max Drawdown $":round(dd,2),
            "Return/Trade %":round(t.ret.mean()*100,2)}

def filter_window(res, lo=MIN_TRADES, hi=MAX_TRADES, rank_by=RANK_BY):
    # keep only configs that trade lo..hi times; if none, fall back to nearest to the midpoint
    w = res[(res["Total Trades"]>=lo) & (res["Total Trades"]<=hi)].copy()
    if w.empty:
        mid = (lo+hi)/2
        res = res.assign(_d=(res["Total Trades"]-mid).abs())
        w = res.sort_values("_d").head(8).drop(columns="_d").copy()
    w["Score"] = w["Total P&L $"] * (w["Win Rate %"]/100.0)   # blended PnL x win-rate
    asc = rank_by in ("Max Drawdown $",)
    return w.sort_values(rank_by, ascending=asc).reset_index(drop=True)

def plot_trades(d, trades, lines, fill, title, stats):
    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(16,10),sharex=True,gridspec_kw={"height_ratios":[3,1.3]})
    ax1.plot(d.Date, d.Close, color="#222", lw=1.1, label="Close")
    for col,lbl,c in lines:
        ax1.plot(d.Date, d[col], color=c, lw=1.2, label=lbl, alpha=0.9)
    if fill:
        b,tp,lbl = fill; ax1.fill_between(d.Date, d[b], d[tp], color="#2ca02c", alpha=0.15, label=lbl)
    if not trades.empty:
        ax1.scatter(trades.entry_date, trades.entry_price, marker="^", s=120, color="green", edgecolor="k", zorder=5, label="BUY")
        ax1.scatter(trades.exit_date,  trades.exit_price,  marker="v", s=120, color="red",   edgecolor="k", zorder=5, label="SELL")
        for _,r in trades.iterrows():
            ax1.plot([r.entry_date,r.exit_date],[r.entry_price,r.exit_price], color="green" if r.pnl>0 else "red", lw=1.0, alpha=0.5)
    ax1.set_title(title, fontsize=14, weight="bold"); ax1.set_ylabel("Price ($)")
    ax1.legend(loc="upper left", ncol=2, fontsize=9); ax1.grid(alpha=0.25)
    if not trades.empty:
        cum = trades.pnl.cumsum().values
        ax2.plot(trades.exit_date, cum, color="#1f77b4", marker="o", ms=3, lw=1.3); ax2.axhline(0, color="grey", lw=0.8)
        ax2.fill_between(trades.exit_date, cum, 0, where=cum>=0, color="green", alpha=0.2)
        ax2.fill_between(trades.exit_date, cum, 0, where=cum<0,  color="red",   alpha=0.2)
    ax2.set_ylabel("Cumulative P&L ($)"); ax2.set_xlabel("Date"); ax2.grid(alpha=0.25)
    ax2.xaxis.set_major_locator(mdates.YearLocator()); ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    items = list(stats.items()); half = (len(items)+1)//2
    l1 = "  |  ".join(f"{k}: {v}" for k,v in items[:half]); l2 = "  |  ".join(f"{k}: {v}" for k,v in items[half:])
    fig.text(0.5, 0.005, l1+"\n"+l2, ha="center", va="bottom", fontsize=10, family="monospace",
             bbox=dict(boxstyle="round", fc="#f5f5f5", ec="#999"))
    fig.tight_layout(rect=[0,0.06,1,1]); plt.show()

def trade_log(trades):
    cols = [c for c in ["entry_date","entry_price","exit_date","exit_price","ret","pnl","bars","reason","status"] if c in trades.columns]
    return trades[cols]

def rsi(series, n=14):
    d = series.diff()
    up = d.clip(lower=0).rolling(n).mean()
    dn = (-d.clip(upper=0)).rolling(n).mean()
    rs = up / dn.replace(0, np.nan)
    return 100 - 100/(1+rs)

def bmsb_prep(df, sma_len, ema_len, fast_ema=10):
    d = df.copy()
    d["SMA"] = d["Close"].rolling(sma_len).mean()
    d["EMA"] = d["Close"].ewm(span=ema_len, adjust=False).mean()
    d["band_top"] = d[["SMA","EMA"]].max(axis=1)
    d["band_bot"] = d[["SMA","EMA"]].min(axis=1)
    d["RSI"] = rsi(d["Close"], 14)
    d["fast"] = d["Close"].ewm(span=fast_ema, adjust=False).mean()
    return d.dropna().reset_index(drop=True)

# Entry modes:
#   anticipate -> BUY while BELOW the band when momentum turns up (catch the move early)
#   reclaim    -> BUY when price closes back above the band bottom after being below
#   breakout   -> classic BMSB: BUY when close clears the band top
#   hybrid_e   -> anticipate OR breakout (whichever triggers first)
# Fakeout defence:
#   fakeout-stop -> exit if price falls stop_pct below entry
#   confirm-fail -> exit if the band is not reclaimed within confirm_bars
#   band-loss    -> after reclaiming the band, exit when price loses the band bottom
def bmsb_backtest(df, sma_len, ema_len, mode=ENTRY_MODE, rsi_buy=RSI_BUY,
                  stop_pct=STOP_PCT, confirm_bars=CONFIRM_BARS, capital=CAPITAL):
    d = bmsb_prep(df, sma_len, ema_len)
    trades = []; pos = None
    for i in range(1, len(d)-1):
        r = d.iloc[i]; p = d.iloc[i-1]; nxt = d.iloc[i+1]
        if pos is None:
            ant = (r.Close < r.band_bot and r.RSI < rsi_buy and r.RSI > p.RSI and r.Close > p.Close)
            brk = r.Close > r.band_top
            rec = (p.Close < p.band_bot and r.Close > r.band_bot)
            fire = {"anticipate":ant, "breakout":brk, "reclaim":rec, "hybrid_e":ant or brk}[mode]
            if fire:
                pos = {"entry_date":nxt.Date, "entry_price":nxt.Open, "entry_i":i+1, "reclaimed": bool(r.Close > r.band_bot)}
        else:
            if r.Close > r.band_bot: pos["reclaimed"] = True
            held = i - pos["entry_i"]
            exit_now, reason = False, ""
            if r.Close <= pos["entry_price"]*(1-stop_pct):           exit_now, reason = True, "fakeout-stop"
            elif (not pos["reclaimed"]) and held >= confirm_bars:    exit_now, reason = True, "confirm-fail"
            elif pos["reclaimed"] and r.Close < r.band_bot:          exit_now, reason = True, "band-loss"
            if exit_now:
                ep = nxt.Open; ret = (ep-pos["entry_price"])/pos["entry_price"]
                trades.append({**pos, "exit_date":nxt.Date, "exit_price":ep, "ret":ret,
                               "pnl":ret*capital, "bars":(i+1)-pos["entry_i"], "reason":reason})
                pos = None
    live = None
    if pos is not None:
        last = d.iloc[-1]; ret = (last.Close-pos["entry_price"])/pos["entry_price"]
        trades.append({**pos, "exit_date":last.Date, "exit_price":last.Close, "ret":ret,
                       "pnl":ret*capital, "bars":(len(d)-1)-pos["entry_i"], "reason":"open"})
    tdf = pd.DataFrame(trades)
    return d, tdf

def bmsb_sweep(df, sma_grid, ema_grid, modes, rsi_grid, stop_grid, cbar_grid):
    rows = []
    for sma in sma_grid:
        for ema in ema_grid:
            for m in modes:
                for rb in rsi_grid:
                    for st in stop_grid:
                        for cb in cbar_grid:
                            _, t = bmsb_backtest(df, sma, ema, m, rb, st, cb)
                            s = compute_stats(t)
                            if s["Total Trades"] == 0: continue
                            rows.append({"SMA":sma,"EMA":ema,"Mode":m,"RSIbuy":rb,"Stop":st,"Cbars":cb, **s})
    return pd.DataFrame(rows)

def bmsb_live(df, sma_len, ema_len, mode, rsi_buy, stop_pct, confirm_bars):
    d, t = bmsb_backtest(df, sma_len, ema_len, mode, rsi_buy, stop_pct, confirm_bars)
    r = d.iloc[-1]; p = d.iloc[-2]
    in_pos = (not t.empty) and (t.iloc[-1]["reason"] == "open")
    if in_pos:
        ep = t.iloc[-1]["entry_price"]
        if r.Close <= ep*(1-stop_pct):      rec = "SELL (fakeout stop hit)"
        elif r.Close < r.band_bot:          rec = "SELL (lost the band)"
        else:                                rec = "HOLD (in position - riding the move)"
        state = "IN position"
    else:
        ant = (r.Close < r.band_bot and r.RSI < rsi_buy and r.RSI > p.RSI and r.Close > p.Close)
        brk = r.Close > r.band_top
        rec_ = (p.Close < p.band_bot and r.Close > r.band_bot)
        fire = {"anticipate":ant, "breakout":brk, "reclaim":rec_, "hybrid_e":ant or brk}[mode]
        rec = "BUY (entry signal - act at next open)" if fire else "HOLD (flat - waiting for entry)"
        state = "FLAT"
    return {"date":str(pd.Timestamp(r.Date).date()), "close":round(float(r.Close),2),
            "band_bot":round(float(r.band_bot),2), "band_top":round(float(r.band_top),2),
            "RSI":round(float(r.RSI),1), "state":state, "recommendation":rec}

def run_bmsb(df, tf, grids):
    sw = bmsb_sweep(df, *grids)
    inwin = int(((sw["Total Trades"]>=MIN_TRADES)&(sw["Total Trades"]<=MAX_TRADES)).sum())
    w = filter_window(sw)
    tag = f"{inwin} with {MIN_TRADES}-{MAX_TRADES} trades" + ("" if inwin>0 else f" (none strictly in range -> showing {len(w)} nearest)")
    print(f"\n=== BMSB {tf} : {len(sw)} combos tested, {tag}. Top 5 by {RANK_BY}: ===")
    show = ["SMA","EMA","Mode","RSIbuy","Stop","Cbars","Total Trades","Win Rate %","Total P&L $","Profit Factor","Score"]
    display(w.head(5)[show])
    b = w.iloc[0]
    d, t = bmsb_backtest(df, int(b.SMA), int(b.EMA), b.Mode, int(b.RSIbuy), float(b.Stop), int(b.Cbars))
    title = f"{TICKER} - BMSB {tf} BEST  (SMA{int(b.SMA)}/EMA{int(b.EMA)}, mode={b.Mode}, RSI<{int(b.RSIbuy)}, stop={b.Stop}, confirm={int(b.Cbars)})"
    plot_trades(d, t, [("SMA",f"{int(b.SMA)} SMA","#1f77b4"),("EMA",f"{int(b.EMA)} EMA","#ff7f0e")],
                ("band_bot","band_top","Support Band"), title, compute_stats(t))
    display(trade_log(t))
    return {"sma":int(b.SMA),"ema":int(b.EMA),"mode":b.Mode,"rsi":int(b.RSIbuy),"stop":float(b.Stop),"cbars":int(b.Cbars)}


## Weekly — sweep, top 5 (8–15 trades), chart the best

In [ ]:
best_w = run_bmsb(weekly, 'Weekly', W_GRID)

## Weekly — canonical **20 SMA / 21 EMA** band (shown next to the best)
Same entry/exit logic as the best config, but forced onto the textbook BMSB lengths so you can compare.

In [ ]:
c = best_w
d, t = bmsb_backtest(weekly, 20, 21, c['mode'], c['rsi'], c['stop'], c['cbars'])
title = f"{TICKER} - BMSB Weekly CANONICAL 20 SMA / 21 EMA  (mode={c['mode']}, RSI<{c['rsi']}, stop={c['stop']}, confirm={c['cbars']})"
plot_trades(d, t, [('SMA','20 SMA','#1f77b4'),('EMA','21 EMA','#ff7f0e')], ('band_bot','band_top','Support Band'), title, compute_stats(t))
display(trade_log(t))

## Daily — sweep, top 5 (8–15 trades), chart the best

In [ ]:
best_d = run_bmsb(daily, 'Daily', D_GRID)

## Daily signal — run me every day → BUY / SELL / HOLD
Uses the best **weekly** config (weekly is the standard BMSB timeframe).

In [ ]:
c = best_w
wk_live = get_data(TICKER, '1wk')
sig = bmsb_live(wk_live, c['sma'], c['ema'], c['mode'], c['rsi'], c['stop'], c['cbars'])
print(f"===== {TICKER} BMSB SIGNAL - {sig['date']} =====")
print(f"Close {sig['close']} | Band {sig['band_bot']}-{sig['band_top']} | RSI {sig['RSI']} | {sig['state']}")
print('>>>', sig['recommendation'])